# ⚠请注意，该模型的计算步骤经过严重的简化，与实际交易情况存在较大误差，请勿作为实际投资参考。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft YaHei']


In [ ]:
# 导入数据

df_csi_500 = pd.read_csv('data/CSI500.csv',index_col=0, parse_dates=True)
# index_col=0，可以让我们后续提取当中某一列作为series时仍能保留原dataframe里的第0列也就是日期作为索引
df_csi_300 = pd.read_csv('data/CSI300.csv',index_col=0, parse_dates=True)
df_csi_500.head()
pct_500 = df_csi_500['涨跌幅(%)Change(%)']
pct_300 = df_csi_300['涨跌幅(%)Change(%)']
pct_300.head()

In [ ]:
# 生成具有两个指数涨跌幅的df
df = pd.DataFrame({
    'CSI500': pct_500,
    'CSI300': pct_300
})
df.head()
df.describe()

In [ ]:
# 假设2016年3月1日的各个净值为1.0000，计算得出每日净值的df
df = df / 100
nav = (1 + df).cumprod()
nav.head()

In [ ]:
# 画两个指数的净值曲线：
plt.figure(figsize=(10,5))
plt.plot(nav.index, nav['CSI500'], label='中证500',color = 'pink')
# 可以发现，横坐标根据其日期格式进行了优化
plt.plot(nav.index, nav['CSI300'], label='沪深300',color = 'skyblue')

plt.legend()
plt.title('Index Net Value')
plt.show()


In [ ]:
# 计算最大回撤（以CSI500）为例

# 计算每日回撤
rolling_max = nav['CSI500'].cummax()    # .cummax()，累计最大值，也就是从这个series第一个开始到第n个数中的最大值，作为n的对应值
drawdown = nav['CSI500']/rolling_max - 1
drawdown_max = drawdown.min()    # 最大回撤的数值是一个负数，从数学角度来看应该是最小值
print('最大回撤：',drawdown_max)

# 绘制回撤曲线
plt.figure(figsize=(10,5))
plt.plot(drawdown)
plt.title('回撤曲线')
plt.show()

In [ ]:
# 一些初始参数
initial_cash = 100000     # 初始本金
monthly_invest = 1000     # 每月定投金额
share = 0                 # 持有份额


In [ ]:
# 首先，初始本金全部买入
initial_price = nav.iloc[0]['CSI500']                 # 2016年3月1日的净值
initial_invest = initial_cash / initial_price         # 2016年3月1日用100000元买入中证500后的份额

# 按照每月定投的投资方案的投资日期
monthly_dates = nav.resample('ME').first().index

# 判断是否为定投日，如果在定投日就用1000元买入份额
for date in nav.index:
    if date in monthly_dates:
        price = nav.loc[date,'CSI500']                # 定投日的净值
        share += monthly_invest / price               # 于定投日买入份额，并且不断累加

# 计算最终资产
final_price = nav.iloc[-1]['CSI500']                           # 2026年3月2日的净值
final_money = (share + initial_invest) * final_price           # 此时中证500基金的持仓金额

total_money = final_money                                      # 在2026年3月2日时的总资产
total_invest = initial_cash + monthly_invest * len(monthly_dates)      # 总投入本金
total_return = total_money/total_invest - 1                    # 收益率

print("总资产：",total_money)
print("总投入本金：",total_invest)
print("收益率",total_return)

In [ ]:
# 同样的总投入本金，如果在2016年3月1日直接一次性买入呢？
print('同样的总投入本金，如果在2016年3月1日直接一次性买入，最终收益为：',(total_invest/initial_price)*final_price)

In [ ]:
# 绘制每日资产曲线
share_daily = pd.Series(0.0, index=nav.index)    # 这个序列用来记录每日的份额
for date in nav.index:
    if date in monthly_dates:
        price = nav.loc[date,'CSI500']
        share_bought = monthly_invest / price
        share_daily.loc[date] += share_bought

initial_share = initial_invest/(nav['CSI500'].iloc[0])
equity_curve = (share_daily.cumsum() + initial_share)*nav['CSI500']

plt.figure(figsize=(10,5))
plt.plot(equity_curve,color='orange')
plt.title('Personal Property')
plt.show()


In [ ]:
# 定投策略与指数对比
plt.figure(figsize=(10,5))
plt.plot(nav['CSI500']*(total_invest/initial_cash), label='CSI500')    # 一口气梭哈的总资产曲线
plt.plot(equity_curve / equity_curve.iloc[0], label='Personal Invest')    # 定投策略的曲线
plt.legend()
plt.show()

# 这个代码的一个缺点在于：两条曲线的起点不同，因此对比起来不够直观。但优点在于更具有现实意义，不会形成误导。

## 根据上图我们可以得出两个结论：

1.在一个整体上涨的市场周期里，投入同样多的本金，最开始一口气梭哈的收益会显著大于定投。

2.定投的优势在于，会让你的资产曲线的上涨趋势比梭哈资产曲线更明显，而且更平滑，最大回撤也更小，其现实意义就在于能够让投资者度过市场大幅波动带来的恐慌。